### 1. Setup Inicial
 

In [ ]:
# Importação das bibliotecas

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Configurações de exibição

pd.set_option('display.max_columns', None)   
pd.set_option('display.float_format', '{:.2f}'.format)

print('Bibliotecas carregadas!')


In [ ]:
# Carregamento dos dados brutos

df_raw = pd.read_csv('../data/Base Varejo.csv', sep=';')

print('Dataframe bruto')
print(f'   Linhas x Colunas : {df_raw.shape[0]} x {df_raw.shape[1]}')
print(f'   Nulos totais     : {df_raw.isnull().sum().sum()}')
print(f'   Duplicatas       : {df_raw.duplicated().sum()}')

# Primeiras visualizações 

print('\n--- Informações do Dataframe ---')
display(df_raw.info())

print('\n--- Primeiras linhas do Dataframe ---')
display(df_raw.head(10))


#### Insigts

- A base está estruturada de forma transacional, pois existem valores nas colunas de ID de compra (`CO_ID`) e ID do cliente (`CL_ID`) que se repetem, tendo variação no ID do produto (`PR_ID`). Isso demostra uma jornada de compra.

- As 4 últimas colunas vieram vazias. Eliminação necessária.

- Coluna de DATA precisa ser alterada para o formato `datetime`

### 2. Transformações

In [ ]:
# Primeiro, copiar o dataframe para as transformações
df_limpo = df_raw.copy()

# Eliminar as últimas quatro colunas que estão totalmente vazias
df_limpo = df_limpo.dropna(how='all', axis=1)

# Converter a coluna 'DATA' para datetime
df_limpo['DATA'] = pd.to_datetime(df_limpo['DATA'], format='%d/%m/%Y')

# Visualização das mudanças
print('\n--- Dataframe após tratamentos iniciais ---')
display(df_limpo.head())

In [ ]:
# Renomear os nomes das colunas para melhor visualização
df_limpo = df_limpo.rename(columns={
    'DATA': 'data_venda',
    'CO_ID': 'id_cupom',
    'CL_ID': 'id_cliente',
    'CL_GENERO': 'genero_cliente',
    'CL_EC': 'estado_civil_cliente',
    'CL_FHL': 'faixa_filhos_cliente',
    'CL_SEG': 'segmentacao_cliente',
    'PR_ID': 'id_produto',
    'PR_CAT': 'categoria_produto',
    'PR_NOME': 'nome_produto'
})

# Renomear os valores das colunas 'genero_cliente'
df_limpo['genero_cliente'] = df_limpo['genero_cliente'].replace({'M': 'Masculino', 'F': 'Feminino'})
print('--- Valores da coluna após o tratamento ---')
print(df_limpo['genero_cliente'].unique())

print('\n--- Dataframe após renomeações ---')
display(df_limpo.head(55))

### 3. Limpeza de Nulos e Duplicatas

In [ ]:
# Verificação de nulos por colunas
nulos = df_limpo.isnull().sum()
pct = (nulos / len(df_limpo) * 100).round(1)

print(f'\n--- Nulos por coluna (%) ---')
print(pct)

In [ ]:
# Verificação de duplicatas por coluna

print(f'\nDuplicatas: {df_limpo.duplicated().sum()}')

#Verificas se as duplicadas são reais ou 
df_limpo[df_limpo.duplicated(keep=False)].head(10)

#### Insigths

- Ao analisar as duplicadas, conclui que não são duplicatas reais, mas sim o registo do mesmo item para o mesmo cliente no mesmo cupom. Com isso, decidi manter todos os valores e analisar melhor agrupando na etapa de estatística descritiva

 

### 4. Estatística Descritiva

In [37]:
# Visão descritiva geral
print('--- Análise descritiva da coluna de Número de filhos do cliente ---')
display(df_limpo['faixa_filhos_cliente'].describe())

print(f"Moda da coluna de filhos: {df_limpo['faixa_filhos_cliente'].mode()[0]}")

--- Análise descritiva da coluna de Número de filhos do cliente ---


count   830000.00
mean         1.15
std          1.42
min          0.00
25%          0.00
50%          0.00
75%          2.00
max          4.00
Name: faixa_filhos_cliente, dtype: float64

Moda da coluna de filhos: 0


### 5. Exploração

In [39]:
# Média de filhos por gênero
filhos_por_genero = df_limpo.groupby('genero_cliente')['faixa_filhos_cliente'].mean().reset_index(name='media_filhos')
print('--- Média de filhos por gênero dos clientes ---')
display(filhos_por_genero)

--- Média de filhos por gênero dos clientes ---


,genero_cliente,media_filhos
0,Feminino,1.09
1,Masculino,1.21


In [ ]:
# Agrupamento de compras por cupom
compras = df_limpo.groupby('id_cupom').size().reset_index(name='quantidade_de_produtos') 
print('--- Quantidade de produtos por cupom ---')
display(compras.head(10))

print(f'Total de cupons = {compras.shape[0]}')

In [ ]:
# Itens comprados por segmento de cliente
compras_segmento = df_limpo.groupby('segmentacao_cliente').size().reset_index(name='total_itens_comprados')
print('--- Quantidade de produtos por segmento ---')
display(compras_segmento)